## Model Training

- Starting off by importing our cleaned datasets

In [2]:
import pandas as pd
import re

In [27]:
kaggle_train_clean = pd.read_csv("./dataset/kaggle_train_clean.csv")
kaggle_test_clean= pd.read_csv("./dataset/kaggle_test_clean.csv")
stack_train = pd.read_csv("./dataset/stack_train.csv")

### Text Preprocessing for NLP

Doing text preprocessing for NLP. As we are using BERT, heavy text preprocessing is not required. We will define functions to remove extra punctuations, handle whitespaces. Then, we do label encoding for the priorities. To make sure that this consistent across all databases, we will define functions to implement these. 

**Function to handle process text:**

In [4]:
import re

def remove_extra_whitespace(text):
    # Replace multiple spaces/tabs/newlines with a single space
    text = re.sub(r'\s+', ' ', text)
    return text.strip()
def to_lowercase(text):
    return text.lower()
    
def preprocess_text(text):
    text = remove_extra_whitespace(text)
    text = to_lowercase(text)
    return text



In [5]:
kaggle_train_clean["clean_query"] = kaggle_train_clean["raw_text"].apply(preprocess_text)
kaggle_test_clean["clean_query"] = kaggle_test_clean["raw_text"].apply(preprocess_text)

In [6]:
label_map = {
    "Low": 0,
    "Medium": 1,
    "High": 2
}

kaggle_train_clean["label"] = kaggle_train_clean["priority_label"].map(label_map)
kaggle_test_clean["label"] = kaggle_test_clean["priority_label"].map(label_map)

In [28]:
from sklearn.model_selection import train_test_split

stack_train["clean_query"] = stack_train["raw_text"].apply(preprocess_text)
stack_train["label"] = stack_train["priority_label"].map(label_map)

We do not need a lot of text preprocessing as BERT does not require it. We will now train and test our BERT model. We will train 2 models, one on the Kaggle dataset and the other on the StackOverflow dataset. We will the compare the cross-accuracy of one on the other, v.s. the same train-test model. Essentially, we are trying to test is the model is acutally learning anything and hence is transferablle or if model is just memorising patterns.

# Tokenization

Tokenization is the process of breaking raw text into smaller units (tokens), such as words or subwords, so a model can process them numerically.
- A BERT tokenizer specifically uses subword tokenization (WordPiece) to split words into known pieces, helping handle rare or unseen words.
- It also converts tokens into input IDs, adds special tokens like [CLS] and [SEP], and creates attention masks for the model.
- This step is necessary because neural networks can only work with numerical inputs, not raw text.
- Attention masks are binary vectors used to tell the model which tokens in an input sequence should be attended to and which should be ignored (usually padding tokens).
- We add a bunch of pads at the end of shorter sentences, so they can be the same length as some of the longer sentences. These pads are ignored while model training.

In [13]:
from utils import tokenize

In [14]:
kaggle_encodings, kaggle_labels = tokenize(kaggle_train_clean, text_column="clean_query", tokenizer=tokenizer, labels="label")

In [15]:
stack_encodings, stack_labels = tokenize(stack_train, text_column="clean_query", tokenizer=tokenizer, labels="label")

## Building the pytorch dataset pipeline

In [16]:
from torch.utils.data import Dataset, DataLoader
import torch

class QueryDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels.iloc[idx])
        return item

kaggle_dataset = QueryDataset(kaggle_encodings, kaggle_labels)

In [19]:
kaggle_dataset[0]

{'input_ids': tensor([ 101, 2129, 2000, 3693, 3076, 4184, 1029,  102,    0,    0,    0,    0,
            0,    0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]),
 'attention_mask': tensor([1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]),
 'labels': tensor(0)}

In [22]:
from torch.utils.data import random_split

train_size = int(0.8 * len(kaggle_dataset))
val_size = len(kaggle_dataset) - train_size
train_data, val_data = random_split(kaggle_dataset, [train_size, val_size])

train_loader = DataLoader(train_data, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_data,   batch_size=16, shuffle=False)

In [ ]:
from transformers import BertForSequenceClassification
import torch

num_labels = kaggle_labels.nunique()f

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=num_labels
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [24]:
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

epochs = 3
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)

In [25]:
from torch.nn import CrossEntropyLoss

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Train Loss: {total_loss / len(train_loader):.4f}")

Epoch 1 | Train Loss: 0.2114
Epoch 2 | Train Loss: 0.0010
Epoch 3 | Train Loss: 0.0006


In [26]:
from sklearn.metrics import accuracy_score

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Val Accuracy: {acc:.4f}")

Val Accuracy: 1.0000


## Model training on stack overflow data

In [31]:
stack_dataset = QueryDataset(stack_encodings, stack_labels)

In [32]:
stack_train_size = int(0.8 * len(stack_dataset))
stack_val_size = len(stack_dataset) - stack_train_size
stack_train_data, stack_val_data = random_split(stack_dataset, [stack_train_size, stack_val_size])

stack_train_loader = DataLoader(stack_train_data, batch_size=16, shuffle=True)
stack_val_loader   = DataLoader(stack_val_data,   batch_size=16, shuffle=False)

In [33]:
stack_num_labels = stack_labels.nunique()

stack_model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=stack_num_labels
)
stack_model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [34]:
stack_optimizer = AdamW(stack_model.parameters(), lr=2e-5, weight_decay=0.01)

stack_total_steps = len(stack_train_loader) * epochs

stack_scheduler = get_linear_schedule_with_warmup(
    stack_optimizer,
    num_warmup_steps=int(0.1 * stack_total_steps),
    num_training_steps=stack_total_steps
)

In [35]:
for epoch in range(epochs):
    stack_model.train()
    stack_total_loss = 0

    for batch in stack_train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        stack_optimizer.zero_grad()

        outputs = stack_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(stack_model.parameters(), max_norm=1.0)

        stack_optimizer.step()
        stack_scheduler.step()

        stack_total_loss += loss.item()

    print(f"Epoch {epoch+1} | Stack Train Loss: {stack_total_loss / len(stack_train_loader):.4f}")

Epoch 1 | Stack Train Loss: 0.6662
Epoch 2 | Stack Train Loss: 0.3884
Epoch 3 | Stack Train Loss: 0.2893


In [36]:
stack_model.eval()
stack_all_preds, stack_all_labels = [], []

with torch.no_grad():
    for batch in stack_val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = stack_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        stack_all_preds.extend(preds.cpu().numpy())
        stack_all_labels.extend(labels.cpu().numpy())

stack_acc = accuracy_score(stack_all_labels, stack_all_preds)
print(f"Stack Val Accuracy: {stack_acc:.4f}")

Stack Val Accuracy: 0.9187


In [2]:
from huggingface_hub import login
login() 

In [41]:
# kaggle model
model.save_pretrained('./kaggle_bert_model')
tokenizer.save_pretrained('./kaggle_bert_model')
model.push_to_hub('nemoonyte/kaggle-bert-model')
tokenizer.push_to_hub('nemoonyte/kaggle-bert-model')

# stack model
stack_model.save_pretrained('./stack_bert_model')
tokenizer.save_pretrained('./stack_bert_model')
stack_model.push_to_hub('nemoonyte/stack-bert-model')
tokenizer.push_to_hub('nemoonyte/stack-bert-model')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/nemoonyte/stack-bert-model/commit/c76abfcce78d5f373040d8b95ade9b60bc3ce84c', commit_message='Upload tokenizer', commit_description='', oid='c76abfcce78d5f373040d8b95ade9b60bc3ce84c', pr_url=None, repo_url=RepoUrl('https://huggingface.co/nemoonyte/stack-bert-model', endpoint='https://huggingface.co', repo_type='model', repo_id='nemoonyte/stack-bert-model'), pr_revision=None, pr_num=None)